# 00 · Your first NexQus circuit / စတင်အသုံးပြုခြင်း

**Goal:** install NexQus, run a Bell circuit, and distinguish probabilities from sampled counts.
**Time:** about 15 minutes. No physics or GPU background is required. Targets NexQus SDK 1.4.0rc1.

NexQus ဖြင့် မိမိကွန်ပျူတာပေါ်တွင် quantum circuit ကို simulation လုပ်နိုင်ပါသည်။
စတင်ရန် quantum hardware သို့မဟုတ် GPU မလိုအပ်ပါ။

Open a terminal in the extracted SDK folder, activate your environment, and run:

```bash
python -m pip install -e ".[tutorial]"
python -m ipykernel install --user --name nexqus --display-name "Python (NexQus)"
jupyter lab notebooks
```

Choose **Python (NexQus)** in Jupyter. Install commands belong in your terminal;
the code cells below contain plain Python, so they also work with the bundled runner.


## Explore NexQus 1.4

- **Circuit/QNN inspection:** run `python examples/explain_circuits.py` and open
  the offline reports. Compare predictions, local sensitivities and recorded
  training checkpoints. [Guide](../docs/EXPLAINABILITY.md).
- **Text to circuit:** run `python examples/nlp_similarity.py` for a local
  sentence-pair laboratory. [Guide](../docs/NLP_SIMILARITY.md).
- **Train a small NLP experiment:** `python examples/train_sentence_pairs.py`.
  [Feature studies and public-data preparation](../docs/NLP.md).
- **Reusable SDK APIs:** `Circuit.explain`, `QuantumDistribution`, and
  `nexqus.nlp` sentence encoders, scalers and parser adapters. Tokenization,
  dataset preparation, training protocols and local web apps are example code.
- **Validation:** CPU/GPU commands and interpretation are in
  [GPU.md](../docs/GPU.md) and [RELEASING.md](../docs/RELEASING.md).

This is release candidate 1.4.0rc1. GPU checks requiring actual CUDA hardware
remain separate from CPU checks. NLP scores are experiment results, not a
language-model or quantum-advantage guarantee.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.4.0rc1', 'Install NexQus SDK 1.4.0rc1 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


## Circuit → simulator → result

A circuit records operations. A simulator executes them. A result contains probabilities and sampled counts.
`q0` is the **leftmost** bit. `Circuit(2).x(0)` prepares `10`.

Circuit သည် gate အစဉ်ကို သိမ်းထားပြီး `run()` က တွက်ချက်ပေးပါသည်။
Bitstring ကို ဘယ်ဘက်မှ q0, q1, ... အစဉ်ဖြင့် ဖတ်ပါ။


In [ ]:
q = Circuit(2).x(0)
result = Simulator().run(q, shots=100, seed=42)
print(result.counts)
assert result.counts == {'10': 100}


In [ ]:
# H creates superposition; CX entangles the two qubits.
# qubit နှစ်ခုကို entangle လုပ်ပါ။
bell = algorithms.bell()
result = Simulator().run(bell, shots=1000, seed=42)
print('Probabilities:', result.probabilities)
print('Counts:', result.counts)
np.testing.assert_allclose(result.probabilities, [.5, 0, 0, .5], atol=1e-12)
assert set(result.counts) <= {'00', '11'}
assert sum(result.counts.values()) == 1000


Probabilities describe the simulated distribution. Counts fluctuate because we sample a finite number of shots.
More shots do not add qubits or make an approximate noise ensemble exact.
`seed` makes these draws repeatable under a fixed SDK/environment configuration.

Probability နှင့် count ကို ခွဲခြားကြည့်ပါ။ Count သည် random sampling ကြောင့် အနည်းငယ်ကွာနိုင်ပါသည်။


In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(['00', '01', '10', '11'], result.probabilities, color='#3b82f6')
plt.ylim(0, 1); plt.ylabel('Probability'); plt.title('Bell-state distribution')
plt.tight_layout(); plt.show()
print('Checks passed. Next: 01_qubits_and_gates.ipynb')


## Draw your own circuit / Circuit ပုံဆွဲခြင်း

SDK 1.4.0rc1 draws any ordinary circuit without simulating it. Here RX and RY
are rotations in radians; `cx` is CNOT. All eight wires appear, including idle
wires. `draw()` returns terminal text; `draw('svg')` displays in Jupyter.
To save a vector file, use `custom.draw('svg', filename='custom.svg')`.
Built-in rotations display numerical angles; symbolic gates include their
expressions. Lesson 07 covers inspection, export and QML helpers.

မိမိတည်ဆောက်ထားသော circuit ကို simulation မလုပ်ဘဲ ပုံဆွဲနိုင်ပါသည်။
Jupyter တွင် SVG ပုံကို ကြည့်နိုင်ပြီး file အဖြစ်လည်း သိမ်းနိုင်ပါသည်။


In [ ]:
custom = Circuit(8).rx(.5, 0).cx(0, 1).ry(.8, 3).cx(3, 7)
text_diagram = custom.draw()
assert all(f'q{i}' in text_diagram for i in range(8))
print(text_diagram)
svg_diagram = custom.draw('svg')
from xml.etree import ElementTree
assert ElementTree.fromstring(svg_diagram).tag.endswith('svg')
svg_diagram  # Jupyter displays SVG; the standalone runner prints the text above.


**Try it:** remove CX and inspect the bitstrings. Then change `shots` from 1000 to 100.
Predict which probabilities change and which differences are only sampling variation.
